# Bootstrap CI for ML Model Metrics

A common problem: you trained a model, got `AUC = 0.91` — but **what is the confidence interval?**
Standard k-fold cross-validation gives variance across folds, not a CI for the metric itself.

This notebook shows how to use `bootstrapx` to get statistically valid CIs for:
- AUC-ROC
- F1-score
- Any custom metric

And demonstrates the `BootstrapCV` scikit-learn adapter.

In [1]:
# !pip install bootstrapx-lib[pandas] scikit-learn
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import cross_val_score, StratifiedKFold

from bootstrapx import bootstrap, BootstrapCV

print(f'bootstrapx ready')

bootstrapx ready


## 1. Bootstrap CI on predictions

Train once, bootstrap the test set predictions to get CI for AUC.

In [2]:
X, y = load_breast_cancer(return_X_y=True)
rng = np.random.default_rng(42)

# Single train/test split
idx = rng.permutation(len(X))
X_train, X_test = X[idx[:400]], X[idx[400:]]
y_train, y_test = y[idx[:400]], y[idx[400:]]

model = GradientBoostingClassifier(n_estimators=100, random_state=0)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]

# Bootstrapx currently expects 1-D data for standard iid methods.
# For metric bootstrapping, resample indices and compute the metric on those indices.
index_data = np.arange(len(y_test))

def auc_statistic(index_sample):
    idx = np.asarray(index_sample, dtype=int)
    return roc_auc_score(y_test[idx], y_prob[idx])

result = bootstrap(
    index_data,
    auc_statistic,
    method='bca',
    n_resamples=4999,
    random_state=42,
)
print(result)
print(f'AUC = {result.theta_hat:.4f}  95% CI: [{result.confidence_interval.low:.4f}, {result.confidence_interval.high:.4f}]')


BootstrapResult(method='bca', theta_hat=0.989214, se=0.00888995, CI=[0.944377, 0.998786])
AUC = 0.9892  95% CI: [0.9444, 0.9988]


## 2. BootstrapCV — OOB cross-validation

More stable estimate using out-of-bag splits.

In [3]:
cv_bootstrap = BootstrapCV(n_splits=200, random_state=0)
cv_kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

scores_boot = cross_val_score(
    GradientBoostingClassifier(n_estimators=100, random_state=0),
    X, y, cv=cv_bootstrap, scoring='roc_auc'
)
scores_kfold = cross_val_score(
    GradientBoostingClassifier(n_estimators=100, random_state=0),
    X, y, cv=cv_kfold, scoring='roc_auc'
)

print(f'Bootstrap CV: {scores_boot.mean():.4f} ± {scores_boot.std():.4f}')
print(f'10-Fold CV:   {scores_kfold.mean():.4f} ± {scores_kfold.std():.4f}')

Bootstrap CV: 0.9893 ± 0.0060
10-Fold CV:   0.9924 ± 0.0086


## 3. pandas accessor on evaluation DataFrame

In [4]:
import bootstrapx  # registers .bootstrap accessor

results_df = pd.DataFrame({
    'bootstrap_auc': scores_boot,
    'kfold_auc': np.resize(scores_kfold, len(scores_boot))
})

summary = results_df.bootstrap.summary(np.mean, n_resamples=2999, random_state=42)
print(summary)

               theta_hat    ci_low   ci_high        se method
column                                                       
bootstrap_auc   0.989341  0.988492  0.990137  0.000423    bca
kfold_auc       0.992361  0.991160  0.993515  0.000603    bca
